In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
MODE = "light"

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools
from torch import nn
from torch import optim


In [ ]:
from torch import nn
from torch import optim


class GroupingAutoencoderFixedSigma(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        recon = self.decoder(codes)
        return recon, codes

    def loss_fn(self, x, recons, codes, alpha, sigma_x, sigma_s, sigma_0):
        recons_loss = self.recon_loss(x, recons, sigma_x)

        codes_loss = self.codes_loss(codes, sigma_s)

        weights_loss = self.weights_loss(alpha, sigma_0)

        return recons_loss, codes_loss, weights_loss

    def codes_loss(self, codes, sigma_s):
        return self.gauss_loss(codes, 0) / (sigma_s * sigma_s)

    def recon_loss(self, x, recons, sigma_x):
        return self.gauss_loss(x, recons) / (sigma_x * sigma_x)

    def cauchy_loss(self, x, cauchy_gamma):
        return torch.log(1 + (x * x) / (cauchy_gamma * cauchy_gamma)).sum()

    def cauchy_codes_loss(self, codes, cauchy_gamma):
        return (
            torch.log(1 + (codes * codes) / (cauchy_gamma * cauchy_gamma)).sum(1).mean()
        )

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        # sum the loss inside one example, send back mean across examples for the batch
        return torch.sum(loss, 1).mean()

    def l1_loss(self, x, mean):
        loss = (x - mean).abs()
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0):
        W = self.decoder.weight
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2

    def masked_recon_loss(self, x, recons, sigma_x):
        mask = (x != 0).float()
        loss = ((x - recons) ** 2) * mask
        return torch.sum(loss, 1).mean() / (sigma_x**2)

    def weights_loss_cycled(self, alpha, sigma_0, chunk_size=64):
        W = self.decoder.weight          # (C, K)
        C, K = W.shape

        idx = (torch.arange(K, device=W.device).unsqueeze(0) +
            torch.arange(K, device=W.device).unsqueeze(1)) % K   # (K, K)

        shift_losses = []

        for start in range(0, K, chunk_size):
            idx_chunk = idx[start:start + chunk_size]   # (chunk, K)

            W_chunk = W[:, idx_chunk]                   # (C, chunk, K)
            W_chunk = W_chunk.permute(1, 0, 2)          # (chunk, C, K)

            W_sq = W_chunk ** 2

            cumsum = torch.cumsum(W_sq, dim=2)
            phi = alpha * torch.roll(cumsum, 1, dims=2) + 1
            phi[:, :, 0] = 1

            comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
            comp2 = -torch.log(phi)

            shift_losses.append((comp1 + comp2).sum(dim=(1, 2)))  # (chunk,)

        return torch.cat(shift_losses).mean()

def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)


def train_grouping_autoencoder_fixed_sigma(
    X,
    n_components,
    max_alpha=5000,
    sigma_x=0.1,
    sigma_s=1,
    sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    weights_algo="cycle",
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_x: std of noise in the data after modelling it as a WS
    sigma_0: std of W, useful to keepn very near 0
    sigma_s: cauchy gamma for cauchy penalty on the encoder (useful for sparse weights)
       the name is sigma_s, cuz it was used as a gaussian prior (L2) on encoder weights, for data which is not sparse
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    # torch.nn.init.normal_(model.decoder.weight, 0, 1)

    # svd, might add back again later
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]

            # first do on recons, let everything flow, make LR for weight 0
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            # alpha = get_alpha(epoch, epochs, 100, max_alpha)
            alpha = max_alpha
            # weights_comp1, weights_log_comp = model.weights_loss(alpha, sigma_0)
            # weight_loss = (weights_comp1 + weights_log_comp).sum()
            if weights_algo == "cycled":
                weight_loss = model.weights_loss_cycled(alpha, sigma_0)
            else:
                comp1, comp2 = model.weights_loss(alpha, sigma_0)
                weight_loss = (comp1 + comp2).sum()

            code_loss = model.gauss_loss(model.encoder[0].weight, 0) / (
                sigma_s * sigma_s
            )

            loss = recon_loss + weight_loss + code_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 200 == 0:
            msg = f"epoch {epoch:4d} | recon_loss {recon_loss:.4f}"
            if weight_loss is not None:
                # msg += f" weight_comp1 {weights_comp1.sum():.4f} weights_log {weights_log_comp.sum():.4f} codes_loss {code_loss:.4f}"
                msg += f" weight_loss {weight_loss.sum():.4f} codes_loss {code_loss:.4f}"
            # msg += f" reg_weight: {reg_weight}"
            print(msg)

    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def train_baseline(X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1):
    # simple linear model without any non linearities
    # for loss checking
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def show_gram(W):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    W_norm = W / (W.norm(dim=0, keepdim=True) + 1e-8)
    gram = W_norm.T @ W_norm  # (n_components, n_components)
    plt.imshow(gram, cmap="gray")
    plt.show()
    return gram

# Oliveti

In [ ]:
n_row, n_col = 1,3
n_components = n_row * n_col
image_shape = (64, 64)

In [ ]:
import logging

import matplotlib.pyplot as plt
from numpy.random import RandomState

from sklearn import cluster, decomposition
from sklearn.datasets import fetch_olivetti_faces

rng = RandomState(0)

# Display progress logs on stdout
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

faces, _ = fetch_olivetti_faces(return_X_y=True, shuffle=True, random_state=rng)
n_samples, n_features = faces.shape

# Global centering (focus on one feature, centering all samples)
faces_centered = faces - faces.mean(axis=0)

# Local centering (focus on one sample, centering all features)
faces_centered -= faces_centered.mean(axis=1).reshape(n_samples, -1)

print("Dataset consists of %d faces" % n_samples)

def plot_gallery(title, images, n_col=n_col, n_row=n_row, cmap=plt.cm.gray):
    fig, axs = plt.subplots(
        nrows=n_row,
        ncols=n_col,
        figsize=(2.0 * n_col, 2.3 * n_row),
        facecolor="white",
        constrained_layout=True,
    )
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.02, hspace=0, wspace=0)
    fig.set_edgecolor("black")
    fig.suptitle(title, size=16)
    for ax, vec in zip(axs.flat, images):
        vmax = max(vec.max(), -vec.min())
        im = ax.imshow(
            vec.reshape(image_shape),
            cmap=cmap,
            interpolation="nearest",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.axis("off")

    fig.colorbar(im, ax=axs, orientation="horizontal", shrink=0.99, aspect=40, pad=0.01)
    plt.show()

plot_gallery("Faces from dataset", faces_centered[:n_components])

In [ ]:
model, codes, components, recon = train_baseline(faces, n_components, 1e-3, 5000)

noise = faces - recon
# mean is very close to zero
print("overall stats", noise.mean(), noise.std())
plt.plot(noise.std(0))
plt.show()

In [ ]:
# linear model is not that great a thing for estimating this
# this is an interesting case, where i know that the noise is not good
# we cannot hope to reconstruct this thing with our model
# what can we do though? linear model is able to catch 7% of the variance
# its best to look at recons though
faces.std(), noise.std()

In [ ]:
from pt_to_api.utils import show

def _r(v):
    return v.reshape(image_shape)

def show_samples(recon, faces):
    for i in range(0, 10, 2):
        show([
            _r(recon[i]),
            _r(faces[i]),
            _r(recon[i+1]),
            _r(faces[i+1]),
        ], (10,5), cmap="gray", ncols=4)

## baseline linear model

the linear model is essentially cheating. It has a baseline shape that it is showing generally.  
Quite interesting what it caught, but nvm.  

So I cant hope to actually match the data at all. So we are fine with larger variance in fitting.  
The problem now is about making sure `W` is still grounded. in that case, `S` mkight do the extra work.  

I'll start with high noise var, but `w` would be smaller (this is not the tested case).  
For our case, we only want to basically do a subset of the image well, those that can be independently drawn. Maybe the model will learn.   
We'll also fix s. Im going ahead with high noise var though

In [ ]:
from pt_to_api.utils import show

def _r(v):
    return v.reshape(image_shape)

for i in range(0, 10, 2):
    show([
        _r(recon[i]),
        _r(faces[i]),
        _r(recon[i+1]),
        _r(faces[i+1]),
    ], (10,5), cmap="gray", ncols=4)

## recons sigma > regularisation sigma

In [ ]:
noise_std = noise.std()
sigma_w = noise_std / 3
sigma_s = sigma_w
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
)

In [ ]:
# interesting lol, its not doing much for reconstruction i guess
# we gave up on reconstruction. and it gave us thisp. Lets see how some of the samples look
# plot_gallery("model", components)

# the first image does contain a lot of "data" but the reconstruction is quite weird
S([c.reshape(image_shape) for c in components], (10,10), ncols=3)
plt.show()
components.shape

In [ ]:
# this is just weird reconstructoin
# and the components dont even have a lot of anything lol
# im assuming we
show_samples(recon, faces)

## recons and regularisation same sigma

weights are again collapsing quite early. Without enough reconstruction pressure it seems to have degenerated i guess.  

In [ ]:
noise_std = noise.std()
sigma_w = noise_std
sigma_s = sigma_w
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
)

In [ ]:
# the first image does contain a lot of "data" but the reconstruction is quite weird
# again, not nice at all. this seems to be the worst ofg them lool
S([c.reshape(image_shape) for c in components], (10,10), ncols=3, viztype="gray")
plt.show()
components.shape

In [ ]:
# its in similar vein though
show_samples(recon, faces)

## much lower recon variance than observed in baseline model (reg = recon)


In [ ]:
noise.std() / 50, faces.std()

In [ ]:
noise_std = noise.std() / 50
sigma_w = noise_std
sigma_s = sigma_w * 5
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
)

In [ ]:
# quite weird again lol
S([c.reshape(image_shape) for c in components], (10,10), ncols=3, viztype="gray")
plt.show()
components.shape

In [ ]:
# why does it pick these extra patches i wonder?
# its in similar vein though
show_samples(recon, faces)

## even lower

In [ ]:
import gc
gc.collect()

In [ ]:
noise_std = noise.std() / 100
sigma_w = noise_std
sigma_s = sigma_w * 10
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
)

In [ ]:
# simialr results
S([c.reshape(image_shape) for c in components], (10,10), ncols=3, viztype="gray")
plt.show()
components.shape

In [ ]:
# why does it pick these extra patches i wonder?
# its in similar vein though
show_samples(recon, faces)

## low recon < low w < low codes

In [ ]:
noise_std / 100

In [ ]:
gc.collect()

noise_std = noise.std() / 10
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
)

In [ ]:
# interesting results now
# lets see how each component looks

S([np.abs(c.reshape(image_shape)) for c in components], (10,10), ncols=3, viztype="gray")
plt.show()

print("############################ samples #####################################33")
# why does it pick these extra patches i wonder?
# its in similar vein though
show_samples(recon, faces)

In [ ]:
np.max(components[0])

In [ ]:
plt.imshow(np.abs(components[2].reshape(image_shape)), cmap="gray")

Might be useful to look at the codes, and each reconstruction

In [ ]:
i = 0
plot_gallery("", [
    components[0]*codes[i][0],
    components[1]*codes[i][1],
    components[2]*codes[i][2],
    recon[i],
    faces[i],
], 5, 1)


In [ ]:
i = 1
plot_gallery("", [
    components[0]*codes[i][0],
    components[1]*codes[i][1],
    components[2]*codes[i][2],
    recon[i],
    faces[i],
], 5, 1)


In [ ]:
# not bad, the penalty is working
show_gram(components.T)

## Try same params with higher components

I might have to increase alpha i think, or tighten the sigma.  
I'll also have to try with shifting the mean in faces to 0 lol

In [ ]:
faces.mean(), faces.min(), faces.max(), faces.std()

In [ ]:
gc.collect()

In [ ]:
n_row, n_col = 2,5
n_components = n_row * n_col

In [ ]:
noise_std = noise.std() / 20
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
)

In [ ]:
# interesting results now
# lets see how each component looks

S([np.abs(c.reshape(image_shape)) for c in components], (20,10), ncols=n_col, viztype="gray")
plt.show()

print("############################ samples #####################################33")
# why does it pick these extra patches i wonder?
# its in similar vein though
show_samples(recon, faces)

In [ ]:
# smaller sigma stabilises gram a bit more
show_gram(components.T)

In [ ]:
i = 1
comps = [components[j]*codes[i][j] for j in range(len(components))]

plot_gallery("each components weight", comps, 5, 2)

plot_gallery("", [recon[i], faces[i]], 2, 1)

## no weights cycling

In [ ]:
n_row, n_col = 2,5
n_components = n_row * n_col
gc.collect()

In [ ]:
noise_std = noise.std() / 20
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
    weights_algo="no-cycle",
)

In [ ]:
# not bad at all

S([np.abs(c.reshape(image_shape)) for c in components], (20,10), ncols=n_col, viztype="gray")
plt.show()

print("############################ samples #####################################33")
# why does it pick these extra patches i wonder?
# its in similar vein though
show_samples(recon, faces)

In [ ]:
# smaller sigma stabilises gram a bit more
show_gram(components.T)

In [ ]:
gc.collect()

### compare with other algos

In [ ]:
ica_estimator = decomposition.FastICA(
    n_components=n_components, max_iter=400, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(faces_centered)

In [ ]:
# not bad at all

S([np.abs(c.reshape(image_shape)) for c in ica_estimator.components_], (20,10), ncols=n_col, viztype="gray")
plt.show()

# print("############################ samples #####################################33")
# # why does it pick these extra patches i wonder?
# # its in similar vein though
# show_samples(recon, faces)

In [ ]:
show_gram(ica_estimator.components_.T)

In [ ]:
nmf_estimator = decomposition.NMF(n_components=n_components, tol=5e-3)
nmf_estimator.fit(faces)  # original non- negative dataset

In [ ]:
# not bad at all

S([np.abs(c.reshape(image_shape)) for c in nmf_estimator.components_], (20,10), ncols=n_col, viztype="gray")
plt.show()

# print("############################ samples #####################################33")
# # why does it pick these extra patches i wonder?
# # its in similar vein though
# show_samples(recon, faces)

In [ ]:
show_gram(nmf_estimator.components_.T)

# original paper, 49 components

In [ ]:
gc.collect()
n_row, n_col = 7,7
n_components = n_row * n_col

In [ ]:
noise_std = noise.std() / 20
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
# tis taking a while with cycle, using no-cycle to see if we get anything usefl
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
    weights_algo="no-cycle",
)

In [ ]:

S([np.abs(c.reshape(image_shape)) for c in components], (20,10), ncols=n_col, viztype="gray")
plt.show()

print("############################ samples #####################################33")
show_samples(recon, faces)

we see the same behavior as before, where stuff is patched. lets decrease sigma_w even more xD

# even less sigma w

In [ ]:
gc.collect()
n_row, n_col = 7,7
n_components = n_row * n_col

In [ ]:
noise_std = noise.std() / 50
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
# tis taking a while with cycle, using no-cycle to see if we get anything usefl
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
    weights_algo="no-cycle",
)

In [ ]:

S([np.abs(c.reshape(image_shape)) for c in components], (20,30), ncols=n_col, viztype="gray")
plt.show()

print("############################ samples #####################################33")
show_samples(recon, faces)

In [ ]:
len(components)

In [ ]:
i = 1
comps = [components[j]*codes[i][j] for j in range(len(components))]

plot_gallery("each components weight", comps, 7, 7)

plot_gallery("", [recon[i], faces[i]], 2, 1)

# still patches, even more less

In [ ]:
gc.collect()
n_row, n_col = 7,7
n_components = n_row * n_col

In [ ]:
noise_std = noise.std() / 100
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
# tis taking a while with cycle, using no-cycle to see if we get anything usefl
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=5000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
    weights_algo="no-cycle",
)

In [ ]:
S([np.abs(c.reshape(image_shape)) for c in components], (20,20), ncols=n_col, viztype="gray")
plt.show()

print("############################ samples #####################################33")
show_samples(recon, faces)

# still patches, not sure if it is a feature

we keep the noise level to the one which gave good results initially, decrease the W sigma compared to before only

In [ ]:
noise_std = noise.std() / 20
sigma_w = noise_std * 2
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces,
    n_components,
    # it bounces wildly xD
    max_alpha=2500 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=5_000,
    weights_algo="no-cycle",
)

In [ ]:
S([np.abs(c.reshape(image_shape)) for c in components], (20,20), ncols=n_col, viztype="gray")
plt.show()

print("############################ samples #####################################33")
show_samples(recon, faces)

# try on negative data now

In [ ]:
# interesting, this is already data with neg values
faces_centered.min()

In [ ]:
ica_estimator = decomposition.FastICA(
    n_components=n_components, max_iter=10_000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(faces_centered)

S([np.abs(c.reshape(image_shape)) for c in ica_estimator.components_], (20,10), ncols=n_col, viztype="gray")
plt.show()

In [ ]:
ica_codes = ica_estimator.transform(faces_centered)
ica_recon = ica_estimator.inverse_transform(codes)

In [ ]:
print("############################ samples #####################################33")
# why does it pick these extra patches i wonder?
# its in similar vein though
show_samples(ica_recon, faces_centered)

In [ ]:
noise_std = noise.std() / 20
sigma_w = noise_std * 5
sigma_s = sigma_w * 3
sigma_x = noise_std
print("s", sigma_s, "w", sigma_w, "noise", noise_std)


# the collapse is quite fast for our weights
model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
    faces_centered,
    n_components,
    # it bounces wildly xD
    max_alpha=3000 * (1 / (sigma_w*sigma_w)),
    sigma_x=sigma_x,
    sigma_s=sigma_s,
    sigma_0=sigma_w,
    lr=1e-3,
    epochs=3_000,
    weights_algo="no-cycle",
)

In [ ]:
S([np.abs(c.reshape(image_shape)) for c in components], (20,10), ncols=n_col, viztype="gray")
plt.show()

In [ ]:
print("############################ samples #####################################33")
show_samples(recon, faces_centered)

There are some patches, im not sure what it is though. anyways. onto topic modelling now.  
That would require a sparser encoder than we have rn, but the penalty cant be on the codes directly as it hurts reconstruction loss.  